## Imports

In [1]:
import rerun as rr
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt 
import csv


from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.geometry import Point

## Read in data

In [2]:
# read in samples as separate csvs 
import glob


path = r"/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Analysis/01-phenotyping/tribus/labels_PanCK_0_aSMA_1_pRb_combined_PH3_gated_600"
all_files = glob.glob(path + "/*.csv")

df_list = []

for filename in all_files:
    #read only the ones that end with "raw_tribus_annotated.csv
    if not filename.endswith("raw_tribus_annotated_pRb_combined_PH3_gated.csv"):
        continue
    df = pd.read_csv(filename)

    image_id = "_".join(os.path.basename(filename).replace(".csv", "").split("_")[0:2])
    image_id = image_id.replace("4-", "")  

    df["imageid"] = image_id
    df_list.append(df)


df = pd.concat(df_list, ignore_index=True)


/tmp/ipykernel_2058881/720248915.py:14: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/tmp/ipykernel_2058881/720248915.py:14: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


## Visualize with rerun

### Add cells

In [3]:
sample = "S095_iOme2"#CHOOSE SAMPLE



subset_df = df[df["imageid"] == sample].reset_index(drop=True)

rr.spawn(f"{sample}", spawn=True)

labels = subset_df["final_label"].values
unique_labels = sorted(np.unique(labels))

cmap = plt.cm.get_cmap("Set1", len(unique_labels))
label_to_color = {
    label: np.array(cmap(i)[:3])
    for i, label in enumerate(unique_labels)
}
node_colors = np.array([label_to_color[label] for label in labels])

node_positions = (
    subset_df[["X_centroid", "Y_centroid"]]
    .to_numpy(dtype=np.float32)
)
cell_labels = subset_df["final_label"].astype(str).to_numpy()

rr.log(
    f"{sample}/cells",
    rr.Points2D(
        positions=node_positions,
        colors=(node_colors * 255).astype(np.uint8),
        radii=2, # size of points (cells)
        labels=cell_labels,   
    ),
)


TypeError: spawn() got an unexpected keyword argument 'spawn'

### Add ROIs (use this if corners in separate columns)

In [4]:
#read in ROI coordinates

roi_df = pd.read_csv(r"P:\h30492\farkkilab2\9_EyeMT\Data\geomx\batch3\roi_coordinates_cycif\geomx_cycif_coordinates_batch3.csv")

#roi_df = roi_df[roi_df["Sample"]==sample]


#scale coordinates to micrometers (if your cell coordinates are in micrometers)
ratio = 1 # == 1 if both cells and rois are in pixels

roi_df = roi_df[roi_df["Sample"]==sample]
roi_df[["roi_c1_X_cycif","roi_c2_X_cycif","roi_c3_X_cycif","roi_c4_X_cycif"]] = roi_df[["roi_c1_X_cycif","roi_c2_X_cycif","roi_c3_X_cycif","roi_c4_X_cycif"]]*ratio
roi_df[["roi_c1_Y_cycif","roi_c2_Y_cycif","roi_c3_Y_cycif","roi_c4_Y_cycif"]] = roi_df[["roi_c1_Y_cycif","roi_c2_Y_cycif","roi_c3_Y_cycif","roi_c4_Y_cycif"]]*ratio


# Loop over each row / ROI
for _, row in roi_df.iterrows():

    roi_name = row["roi_name"]

    # Extract polygon corner coordinates
    # Ensure the order matches how your table defines the rectangle
    xs = [row["roi_c1_X_cycif"], row["roi_c2_X_cycif"], row["roi_c3_X_cycif"], row["roi_c4_X_cycif"] ,row["roi_c1_X_cycif"]]
    ys = [row["roi_c1_Y_cycif"], row["roi_c2_Y_cycif"], row["roi_c3_Y_cycif"], row["roi_c4_Y_cycif"], row["roi_c1_Y_cycif"]]

    pts = np.column_stack([xs, ys])

   
    rr.log(
        f"{sample}/rois/{roi_name}",
        rr.LineStrips2D([pts])
    )

OSError: [Errno 22] Invalid argument: 'P:\\h30492\\farkkilab2\\9_EyeMT\\Data\\geomx\\batch3\\roi_coordinates_cycif\\geomx_cycif_coordinates_batch3.csv'

### Add ROIs (use this if ROIs in array)

In [ ]:
ratio = 1 # == 1 if both cells and rois are in pixels

path = r"P:\h30492\farkkilab2\9_EyeMT\Data\geomx\batch3\roi_coordinates_cycif\cycif_roi_arrays"

# Read CSV as raw lines
with open(fr"{path}\{sample}.csv" ,"r") as f:
    lines = f.readlines()

# Header
header = lines[0].strip().split(",")
data_lines = lines[1:]

rows = []

for line in data_lines:
    # Split first column as ROI_Name
    roi_name, rest = line.strip().split(",", 1)

    # Use regex to find arrays inside brackets
    array_matches = re.findall(r"\[([^\]]+)\]", rest)

    if len(array_matches) != 2:
        raise ValueError(f"Expected 2 arrays in row: {line}")

    x_str, y_str = array_matches

    # Split by any whitespace or commas and convert to floats
    xs = [float(x) * ratio for x in re.split(r"[\s,]+", x_str.strip()) if x]
    ys = [float(y) * ratio for y in re.split(r"[\s,]+", y_str.strip()) if y]

    rows.append({"ROI_Name": roi_name, "X": xs, "Y": ys})

# Convert to DataFrame
roi_df = pd.DataFrame(rows)

for idx, row in roi_df.iterrows():
    roi_name = row["ROI_Name"]
    xs = row["X"]
    ys = row["Y"]

    pts = np.column_stack([xs, ys])

    rr.log(
        f"{sample}/rois/{roi_name}",
        rr.LineStrips2D([pts])
    )



## Add ROI information to cell data

In [5]:
sample = "S015_iOme"#CHOOSE SAMPLE


df = df[df["imageid"] == sample].reset_index(drop=True)

df["ROI"] = "None"


ratio = 1 # == 1 if both cells and rois are in pixels

path = r"P:\h30492\farkkilab2\9_EyeMT\Data\geomx\batch3\roi_coordinates_cycif\cycif_roi_arrays"

rois_dict = dict()

for sample in df["imageid"].unique():

    rois_dict[sample] = dict()

    # Read CSV as raw lines
    with open(fr"{path}\{sample}.csv" ,"r") as f:
        lines = f.readlines()

    # Header
    header = lines[0].strip().split(",")
    data_lines = lines[1:]

    rows = []

    for line in data_lines:
        # Split first column as ROI_Name
        roi_name, rest = line.strip().split(",", 1)

        # Use regex to find arrays inside brackets
        array_matches = re.findall(r"\[([^\]]+)\]", rest)

        if len(array_matches) != 2:
            raise ValueError(f"Expected 2 arrays in row: {line}")

        x_str, y_str = array_matches

        # Split by any whitespace or commas and convert to floats
        xs = [float(x) * ratio for x in re.split(r"[\s,]+", x_str.strip()) if x]
        ys = [float(y) * ratio for y in re.split(r"[\s,]+", y_str.strip()) if y]

        rows.append({"ROI_Name": roi_name, "X": xs, "Y": ys})

    # Convert to DataFrame
    roi_df = pd.DataFrame(rows)


    from shapely.geometry import Polygon
    from shapely.affinity import scale

    new_rows = []

    for _, row in roi_df.iterrows():
        xs = row["X"]
        ys = row["Y"]
        
        poly = Polygon(zip(xs, ys))
        rect = poly.minimum_rotated_rectangle
        
        if not rect.contains(poly):
            rect = scale(rect, xfact=1.001, yfact=1.001, origin="center")
        
        rect_coords = list(rect.exterior.coords)[:-1]
        
        new_rows.append({
            "ROI_Name": row["ROI_Name"],
            "X": [p[0] for p in rect_coords],
            "Y": [p[1] for p in rect_coords]
        })

    roi_df = pd.DataFrame(new_rows)


    roi_polygons = {}
    for idx, row in roi_df.iterrows():
        roi_name = row["ROI_Name"]
        xs = row["X"]
        ys = row["Y"]

        pts = np.column_stack([xs, ys])
        polygon = Polygon(pts)
        polygon = prep(polygon)
        rois_dict[sample][roi_name] = polygon

for idx, cell in df.iterrows():
    sample = cell["imageid"]
    x = cell["X_centroid"]*ratio
    y = cell["Y_centroid"]*ratio
    point = Point(x, y)

    for roi_name, polygon in rois_dict[sample].items():
        if polygon.contains(point):
            df.at[idx, "ROI"] = roi_name
            break



OSError: [Errno 22] Invalid argument: 'P:\\h30492\\farkkilab2\\9_EyeMT\\Data\\geomx\\batch3\\roi_coordinates_cycif\\cycif_roi_arrays\\S015_iOme.csv'

In [36]:
#count number of each cell type in each ROI

cell_counts_in_roi = dict()

for sample in df["imageid"].unique():
    sample_df = df[df["imageid"] == sample]
    cell_counts_in_roi[sample] = sample_df.groupby("ROI")["final_label"].value_counts().unstack(fill_value=0)

#make dataframe from dictionary cell counts in ROIs

cell_counts_df = pd.DataFrame()
for sample, counts_df in cell_counts_in_roi.items():
    counts_df = counts_df.reset_index()
    counts_df["imageid"] = sample
    cell_counts_df = pd.concat([cell_counts_df, counts_df], ignore_index=True)

#remove the "final_label" column name from the cell type columns
cell_counts_df.columns.name = None


cell_counts_df.head()


,ROI,Immune_CD4_Tcells,Immune_CD8_Tcells,Immune_Dcs,Immune_Macrophages,Stroma_Stroma,Tumor_Tumor,other_Tumor,other_Tumor_Immune,undefined_Global,undefined_Immune,undefined_Stroma,undefined_Stroma_Immune,undefined_Tumor,undefined_Tumor_Immune,imageid
0,1,310,51,40,50,3,0,0,0,0,5,0,1,0,0,S015_iOme
1,10,136,89,34,55,49,0,0,0,0,0,0,0,0,0,S015_iOme
2,11,69,27,13,16,59,0,0,0,0,1,0,4,0,0,S015_iOme
3,12,20,14,18,62,64,14,0,0,0,3,0,0,0,0,S015_iOme
4,13,24,29,15,62,24,0,0,0,0,1,0,1,0,0,S015_iOme
